# Практична робота № 4. Агрегація та групування
**Виконав:** Мельниківський Андрій, група КН-41

**Набір даних:** `sales.csv` — продажі кав'ярні (800 транзакцій, березень–квітень 2024).

## Крок 1. Підготовка середовища та завантаження даних

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Підключення Google Диска (у Colab). Поза Colab — використовується локальна папка.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/'
except ImportError:
    BASE = ''
    print('Colab не виявлено — використовується локальна папка')
PATH = BASE + 'sales.csv'
print('Шлях до файлу:', PATH)

Colab не виявлено — використовується локальна папка
Шлях до файлу: sales.csv


In [3]:
df = pd.read_csv(PATH)
df.head()

,date,datetime,cash_type,card,money,coffee_name
0,2024-03-01,2024-03-01 10:19:37.400,card,ANON-0000-0000-0001,35.99,Cocoa
1,2024-03-01,2024-03-01 12:42:16.125,card,ANON-0000-0000-0002,26.88,Americano
2,2024-03-01,2024-03-01 13:16:18.819,card,ANON-0000-0000-0003,28.90,Americano
3,2024-03-01,2024-03-01 14:09:29.333,card,ANON-0000-0000-0004,35.99,Latte
4,2024-03-01,2024-03-01 15:13:38.566,card,ANON-0000-0000-0005,35.99,Latte


## Крок 2. Групування за однією колонкою та агрегація
`groupby()` лише готує групи; обчислення виконується, коли застосовано агрегатну функцію.

In [4]:
revenue_by_coffee = df.groupby('coffee_name')['money'].sum()
print(revenue_by_coffee)

coffee_name
Americano              2986.02
Americano with Milk    6088.85
Cappuccino             4709.37
Cocoa                  2389.32
Cortado                1563.24
Espresso                469.92
Hot Chocolate          2474.85
Latte                  6993.43
Name: money, dtype: float64


In [5]:
avg_money = df.groupby('coffee_name')['money'].mean()
print(avg_money)

coffee_name
Americano              28.438286
Americano with Milk    33.091576
Cappuccino             37.978790
Cocoa                  37.925714
Cortado                28.422545
Espresso               23.496000
Hot Chocolate          38.074615
Latte                  38.007772
Name: money, dtype: float64


Результат — `Series` з назвами напоїв в індексі. `sum()` показує дохід за напоєм, `mean()` — середню суму транзакції.

## Крок 3. Групування за кількома колонками

In [6]:
trans_count = df.groupby(['date', 'coffee_name']).size()
print(trans_count)

date        coffee_name  
2024-03-01  Americano        4
            Cappuccino       1
            Cocoa            1
            Cortado          1
            Latte            3
                            ..
2024-04-30  Cappuccino       4
            Cocoa            1
            Cortado          1
            Hot Chocolate    2
            Latte            4
Length: 345, dtype: int64


In [7]:
trans_count_df = df.groupby(['date', 'coffee_name']).size().reset_index(name='count')
print(trans_count_df.head(10))

         date          coffee_name  count
0  2024-03-01            Americano      4
1  2024-03-01           Cappuccino      1
2  2024-03-01                Cocoa      1
3  2024-03-01              Cortado      1
4  2024-03-01                Latte      3
5  2024-03-02            Americano      1
6  2024-03-02  Americano with Milk      4
7  2024-03-02           Cappuccino      4
8  2024-03-02              Cortado      1
9  2024-03-02                Latte      2


`size()` повертає `Series` з мультиіндексом (дата, напій); `reset_index(name='count')` перетворює його на звичайну таблицю з трьома колонками.

## Крок 4. Кілька агрегатних функцій через `agg`

In [8]:
summary = df.groupby('coffee_name')['money'].agg(['sum', 'mean', 'min', 'max'])
print(summary)

                         sum       mean    min   max
coffee_name                                         
Americano            2986.02  28.438286  26.88  28.9
Americano with Milk  6088.85  33.091576  31.43  33.8
Cappuccino           4709.37  37.978790  35.99  38.7
Cocoa                2389.32  37.925714  35.99  38.7
Cortado              1563.24  28.422545  26.88  28.9
Espresso              469.92  23.496000  22.32  24.0
Hot Chocolate        2474.85  38.074615  35.99  38.7
Latte                6993.43  38.007772  35.99  38.7


## Крок 5. Зведена таблиця (`pivot_table`)

In [9]:
# Створюємо колонку дня тижня (понеділок = 0, неділя = 6)
df['day_of_week'] = pd.to_datetime(df['datetime']).dt.dayofweek

pivot = pd.pivot_table(df, values='money', index='coffee_name', columns='day_of_week', aggfunc='sum', fill_value=0)
print(pivot)

day_of_week               0       1       2       3       4        5        6
coffee_name                                                                  
Americano            600.84  598.82  338.72  369.64  311.84   309.82   456.34
Americano with Milk  835.52  862.21  837.89  985.56  724.64   925.07   917.96
Cappuccino           729.88  644.35  605.65  613.78  721.75   793.73   600.23
Cocoa                229.49  260.06  575.08  638.93  226.78   229.49   229.49
Cortado              371.66  223.12  111.56  144.50  282.94   227.16   202.30
Espresso             118.32   24.00   24.00   44.64   24.00   142.32    92.64
Hot Chocolate        268.19  345.59  185.37  461.69  605.65   190.79   417.57
Latte                799.15  881.97  987.23  951.24  876.55  1324.69  1172.60


## Крок 6. Перехресна таблиця (`crosstab`)

In [10]:
crosstab = pd.crosstab(df['coffee_name'], df['cash_type'])
print(crosstab)

cash_type            card  cash
coffee_name                    
Americano             103     2
Americano with Milk   179     5
Cappuccino            117     7
Cocoa                  61     2
Cortado                53     2
Espresso               20     0
Hot Chocolate          62     3
Latte                 175     9


Таблиця показує частоту комбінацій «напій × тип оплати»: більшість транзакцій оплачено карткою.

## Крок 7. Об'єднання за допомогою `merge`

In [11]:
categories = pd.DataFrame({
    'coffee_name': ['Latte', 'Americano', 'Hot Chocolate', 'Cappuccino'],
    'category': ['молочний', 'чорний', 'молочний', 'молочний']
})

df_merged = df.merge(categories, on='coffee_name', how='left')
print(df_merged.head())

         date                 datetime cash_type                 card  money  \
0  2024-03-01  2024-03-01 10:19:37.400      card  ANON-0000-0000-0001  35.99   
1  2024-03-01  2024-03-01 12:42:16.125      card  ANON-0000-0000-0002  26.88   
2  2024-03-01  2024-03-01 13:16:18.819      card  ANON-0000-0000-0003  28.90   
3  2024-03-01  2024-03-01 14:09:29.333      card  ANON-0000-0000-0004  35.99   
4  2024-03-01  2024-03-01 15:13:38.566      card  ANON-0000-0000-0005  35.99   

  coffee_name  day_of_week  category  
0       Cocoa            4       NaN  
1   Americano            4    чорний  
2   Americano            4    чорний  
3       Latte            4  молочний  
4       Latte            4  молочний  


In [12]:
print('Рядків без категорії (напоїв немає в categories):', df_merged['category'].isnull().sum())
print(df_merged[df_merged['category'].isnull()]['coffee_name'].unique())

Рядків без категорії (напоїв немає в categories): 322
['Cocoa' 'Cortado' 'Americano with Milk' 'Espresso']


`how='left'` зберігає всі рядки `df`; для напоїв, яких немає в `categories`, значення `category` дорівнює `NaN`.

## Крок 8. Об'єднання за допомогою `concat`

In [13]:
df_part1 = df[df['date'] <= '2024-03-15']
df_part2 = df[df['date'] > '2024-03-15']

df_concat = pd.concat([df_part1, df_part2])
print(f"Розміри: {len(df_part1)} + {len(df_part2)} = {len(df_concat)}")

Розміри: 187 + 613 = 800


## Крок 9. Об'єднання за допомогою `join`

In [14]:
left = pd.DataFrame({'A': [1, 2, 3]}, index=['a', 'b', 'c'])
right = pd.DataFrame({'B': [10, 20, 30]}, index=['a', 'b', 'c'])

joined = left.join(right)
print(joined)

   A   B
a  1  10
b  2  20
c  3  30


## Крок 10. Аналітичні показники за групами: частка доходу

In [15]:
total_revenue = df['money'].sum()
revenue_share = (df.groupby('coffee_name')['money'].sum() / total_revenue * 100).round(2)
print(revenue_share)
print('Сума часток:', revenue_share.sum().round(2))

coffee_name
Americano              10.79
Americano with Milk    22.00
Cappuccino             17.02
Cocoa                   8.63
Cortado                 5.65
Espresso                1.70
Hot Chocolate           8.94
Latte                  25.27
Name: money, dtype: float64
Сума часток: 100.0


Отриманий `Series` показує, який відсоток загального доходу приносить кожен напій; сума часток дорівнює 100%.